In [1]:
# CELL 1: Install RL Dependencies (FIXED)

# Force correct numpy version
!pip uninstall -y numpy
!pip install numpy==1.26.4

# Install compatible versions
!pip install gymnasium stable-baselines3 torch
!pip install tensorboard==2.11.2

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
    --------------------------------------- 0.3/15.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.8 MB 1.2 MB/s eta 0:00:13
   - -------------------------------------- 0.8/15.8 MB 1.1 MB/s eta 0:00:14
   - -------------------------------------- 0.8/15.8 MB 1.1 MB/s eta 0:00:14
   -- ------------------------------------- 1.0/15.8 MB 1.1 MB/s eta 0:00:14
   --- ------------------------------------ 1.6/15.8 MB 1.2 MB/s eta 0:00:12
   ---- ----------------------------------- 1.8/15.8 MB 1.3 MB/s eta 0:00:11
   ----- ---------------------------------- 2.1/15.8 MB 1.3 MB/s eta 0:00:11
   ----- ---------------------------------- 2.4/15.8 MB 1.3 MB/s eta 0:00:11
   ----- ---------------------------------- 2.4/15.8 MB 1.3 MB/s eta 0:00:11
   ------- ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tensorflow 2.10.0 requires tensorboard<2.11,>=2.10, but you have tensorboard 2.11.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ===============================
# IMPORTS
# ===============================
import gymnasium as gym
from gymnasium import spaces
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor


# ===============================
# LOAD DATA
# ===============================
print("Loading 3D Tensors...")

data = np.load('MTech_Tensors.npz')
X_train = data['X_train'].astype(np.float32)
y_train = data['y_train'].astype(np.float32)


# ===============================
# ENVIRONMENT (FINAL FIXED)
# ===============================
class MTechTradingEnv(gym.Env):

    def __init__(self, X, y, transaction_cost=0.0001):
        super().__init__()

        self.X = X
        self.y = y
        self.n_samples = len(X)

        self.transaction_cost = transaction_cost

        # Actions: 0=Short, 1=Flat, 2=Long
        self.action_space = spaces.Discrete(3)

        #  FIXED SHAPE
        self.obs_shape = X.shape[1] * X.shape[2]

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.obs_shape,),
            dtype=np.float32
        )

    # ---------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.current_position = 0

        obs = self.X[self.current_step].flatten()
        return obs.astype(np.float32), {}

    # ---------------------------
    def step(self, action):

        action = int(action)
        signal = action - 1  # -1, 0, 1

        # ---- Trading cost ----
        trade_penalty = 0.0
        hold_bonus = 0.0

        if signal != self.current_position:
            trade_penalty = self.transaction_cost
        elif signal != 0:
            hold_bonus = 0.00002

        self.current_position = signal

        # ---- Market return ----
        market_return = float(self.y[self.current_step])

        #  Safety
        if not np.isfinite(market_return):
            market_return = 0.0

        step_return = (market_return * self.current_position) - trade_penalty + hold_bonus

        #  Stability
        step_return = np.clip(step_return, -0.1, 0.1)

        reward = float(step_return * 100.0)

        # ---- Step forward ----
        self.current_step += 1

        terminated = self.current_step >= self.n_samples - 1
        truncated = False

        if not terminated:
            obs = self.X[self.current_step].flatten()
        else:
            obs = self.X[-1].flatten()

        return obs.astype(np.float32), reward, terminated, truncated, {}


print(" Enhanced Environment with Hold Bonus is ready!")




Loading 3D Tensors...
✅ Enhanced Environment with Hold Bonus is ready!


In [ ]:
# ===============================
# ENV WRAPPING (FIXED)
# ===============================
def make_env():
    return Monitor(MTechTradingEnv(X_train, y_train))


env = DummyVecEnv([make_env])


# ===============================
# MODEL
# ===============================
print(" Initializing Deep Learning Brain...")

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    ent_coef=0.01,
    gamma=0.99,
)


# ===============================
# TRAINING
# ===============================
print(" Starting 1,000,000 Step Training Run...")

model.learn(total_timesteps=1_000_000)

model.save("MTech_PPO_Trading_Agent_Fixed")

print(" Million-Step Training Complete and Saved!")

🤖 Initializing Deep Learning Brain...
Using cpu device
🚀 Starting 1,000,000 Step Training Run...
-----------------------------
| time/              |      |
|    fps             | 1916 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1252        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010420093 |
|    clip_fraction        | 0.0704      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -19.7       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0496     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0187     |
|    value_loss      

In [ ]:
# Ensure data exists
if 'y_test' not in globals():
    data = np.load('MTech_Tensors.npz')
    X_test = data['X_test']
    y_test = data['y_test']

initial_balance = 10000

portfolio_values = [initial_balance]
buy_and_hold_values = [initial_balance]

current_balance = initial_balance
bh_balance = initial_balance

transaction_cost = 0.0001
last_signal = 0

print(f"y_test range: {np.min(y_test):.6f} → {np.max(y_test):.6f}")

# Normalize if needed
if np.max(np.abs(y_test)) > 1:
    print(" Normalizing returns (/100)")
    y_test = y_test / 100.0

for i in range(len(X_test)):
    obs = X_test[i].flatten().astype(np.float32)
    action, _ = model.predict(obs, deterministic=True)
    action = int(action)

    signal = action - 1

    market_return = float(y_test[i])
    if not np.isfinite(market_return):
        market_return = 0.0

    fee = transaction_cost if signal != last_signal else 0.0

    step_return = (market_return * signal) - fee
    step_return = np.clip(step_return, -0.1, 0.1)

    current_balance *= (1 + step_return)
    current_balance = max(current_balance, 1)

    portfolio_values.append(current_balance)

    bh_balance *= (1 + market_return)
    bh_balance = max(bh_balance, 1)

    buy_and_hold_values.append(bh_balance)

    last_signal = signal

print(" Backtest complete.")

y_test range: -0.007676 → 0.005432
✅ Backtest complete.
